# 3. Choosing the threshold, and what to trigger on

[Notebook 2](02-triggered-interventions.ipynb) used one threshold, 350 per 100,000. A policy
question is usually *which* threshold. Because the threshold is just a parameter of one
compiled model, summer4 can answer that with JAX's tools:

1. **`jax.vmap`** runs a whole sweep of thresholds as one vectorised program.
2. **`jax.grad`** differentiates outcomes with respect to the threshold, through the trigger.
3. **Two-way sweeps**, here threshold by roll-out time.
4. **A different signal.** A programme cannot see true prevalence continuously, but it does
   see notifications. The trigger signal is an argument to `build_model`, so switching to
   notifications is a one-line change.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import time

import jax
import jax.numpy as jnp
import plotly.express as px

import tb_trigger.model as tbm

cm = tbm.build_model().compile()
baseline = tbm.indicators(tbm.run(cm, tbm.scenario_params("baseline")))

## 1. Sweep the threshold with `vmap`

`sweep_run(threshold)` runs the `acf_triggered` scenario for one threshold. `jax.vmap` maps it
over 47 thresholds from 250 to 480 per 100,000, and `jax.jit` compiles the batch once. The
result is a single summer4 `Result` whose arrays have a leading threshold axis.

The baseline peaks at about 475 per 100,000, so thresholds above that never fire.

In [ ]:
ts = np.round(np.arange(2015.0, 2050.01, 0.1), 1)
plan = tbm.save_plan(ts)
base_params = tbm.scenario_params("acf_triggered")


def sweep_run(threshold):
    return cm.run({**base_params, "threshold": threshold}, save=plan, **tbm.RUN_KWARGS)


thresholds = jnp.linspace(250.0, 480.0, 47)
start = time.perf_counter()
swept = jax.jit(jax.vmap(sweep_run))(thresholds)
print(f"{len(thresholds)} runs in {time.perf_counter() - start:.1f} s, compile time included")

comp = np.asarray(swept["compartments"].values.data)  # (threshold, time, compartment)
people = cm.pmap.select(tbm.PEOPLE)
prevalence = 1e5 * comp[:, :, cm.pmap.select(tbm.people("I"))].sum(-1) / comp[:, :, people].sum(-1)
waiting = comp[:, :, cm.pmap.select(tbm.programme("waiting"))].sum(-1)
deaths = np.asarray(swept["tb_death"].values.data).sum(-1)

### Prevalence trajectories across thresholds

One line per threshold, every fourth shown. Each line follows the baseline until it reaches its
own threshold and turns over within a couple of years. By 2045 they have all settled at the
same post-ACF level. Low thresholds dip below it first, because ACF starts before the rise in
transmission is complete. The threshold changes *when* control starts, not where prevalence
ends up. Lines for
thresholds above the baseline peak follow the baseline throughout.

In [ ]:
lines = pd.DataFrame(prevalence[::4].T, index=pd.Index(ts, name="year"), columns=[f"{t:.0f}" for t in np.asarray(thresholds)[::4]])
fig = lines.plot(
    title="Prevalence under ACF triggered at different thresholds",
    labels={"value": "prevalence per 100,000", "variable": "threshold"},
    color_discrete_sequence=px.colors.sequential.Viridis,
)
fig.add_scatter(x=baseline.loc[2015:].index, y=baseline.loc[2015:, "prevalence"], name="baseline", line=dict(color="black", dash="dash"))
fig

### Trigger year and deaths against the threshold

The left panel shows the trigger year, the year half the programme has left `waiting`, which
follows the baseline's prevalence curve. The right panel shows cumulative TB deaths from 2025
to 2050. Every 10 per 100,000 of extra threshold costs lives, up to the baseline peak, beyond
which the programme never runs and deaths equal the baseline (dashed line).

In [ ]:
from plotly.subplots import make_subplots

trigger_year = [tbm.crossing_year(pd.Series(1.0 - w, index=ts), 0.5) for w in waiting]
window = (ts >= 2025.0) & (ts <= 2050.0)
cum_deaths = np.trapezoid(deaths[:, window], ts[window], axis=1)
base_deaths = float(np.trapezoid(baseline.loc[2025.0:2050.0, "tb_deaths"], baseline.loc[2025.0:2050.0].index))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Trigger year", "TB deaths 2025–2050 per 100,000"))
fig.add_scatter(x=np.asarray(thresholds), y=trigger_year, mode="lines+markers", name="trigger year", row=1, col=1)
fig.add_scatter(x=np.asarray(thresholds), y=cum_deaths, mode="lines+markers", name="cumulative TB deaths", row=1, col=2)
fig.add_hline(y=base_deaths, line_dash="dash", annotation_text="baseline", row=1, col=2)
fig.update_xaxes(title_text="threshold (prevalence per 100,000)")
fig.update_layout(title="What the threshold choice costs", showlegend=False)
fig

## 2. Differentiate through the trigger

The switch is smooth, so cumulative deaths are a differentiable function of the threshold.
`jax.grad` gives the marginal cost of raising it: extra TB deaths per 100,000 for each extra
1 per 100,000 of threshold. The figure draws that gradient as a tangent line at 350. It should
lie along the sweep curve.

Gradients like this are what summer4's calibration and optimisation tools use, so a
threshold, or any intervention parameter, can be fitted or optimised directly.

In [ ]:
def cumulative_deaths(threshold):
    result = sweep_run(threshold)
    series = jnp.ravel(result["tb_death"].total().values)
    keep = (ts >= 2025.0) & (ts <= 2050.0)
    return jnp.trapezoid(series[keep], ts[keep])


at = 350.0
value, slope = jax.jit(jax.value_and_grad(cumulative_deaths))(at)
print(f"deaths 2025–2050 at threshold {at:g}: {float(value):.0f} per 100,000")
print(f"d(deaths)/d(threshold): {float(slope):.2f} deaths per unit of threshold")

window_t = np.array([at - 40.0, at + 40.0])
fig = px.line(x=np.asarray(thresholds), y=cum_deaths, markers=True, labels={"x": "threshold (per 100,000)", "y": "TB deaths 2025–2050 per 100,000"}, title="jax.grad of deaths with respect to the threshold (tangent at 350)")
fig.add_scatter(x=window_t, y=float(value) + float(slope) * (window_t - at), name="tangent from jax.grad", mode="lines", line=dict(dash="dash", width=4, color="black"))
fig.add_scatter(x=[at], y=[float(value)], name="threshold 350", mode="markers", marker=dict(size=12, color="red"))
fig

## 3. Threshold × roll-out time

Nesting two `vmap`s gives a grid. Here it is 24 thresholds by 12 roll-out times in one
compiled call. The colour shows TB deaths averted from 2025 to 2050 compared with the
baseline. Read it as a trade-off between triggering early and scaling up fast. Triggering at
300 with a 6-year roll-out averts about as many deaths (~670 per 100,000) as triggering at 400
with a 3-month roll-out (~650).

In [ ]:
def averted(threshold, rollout_years):
    result = cm.run({**base_params, "threshold": threshold, "rollout_years": rollout_years}, save=plan, **tbm.RUN_KWARGS)
    series = jnp.ravel(result["tb_death"].total().values)
    keep = (ts >= 2025.0) & (ts <= 2050.0)
    return base_deaths - jnp.trapezoid(series[keep], ts[keep])


grid_thresholds = jnp.linspace(250.0, 480.0, 24)
grid_rollout = jnp.linspace(0.25, 6.0, 12)
start = time.perf_counter()
grid = jax.jit(jax.vmap(jax.vmap(averted, in_axes=(None, 0)), in_axes=(0, None)))(grid_thresholds, grid_rollout)
print(f"{grid.size} runs in {time.perf_counter() - start:.1f} s, compile time included")

px.imshow(
    np.asarray(grid).T,
    x=np.round(np.asarray(grid_thresholds), 0),
    y=np.round(np.asarray(grid_rollout), 2),
    origin="lower",
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "threshold (prevalence per 100,000)", "y": "mean roll-out time (years)", "color": "deaths averted"},
    title="TB deaths averted 2025–2050 per 100,000",
)

## 4. Trigger on notifications instead

Prevalence surveys are rare, but notifications arrive every quarter. The model can trigger on
the **notification rate**. `tbm.notifications_per_100k()` is another rate expression, built
from the current values of the two detection flows:

```python
def notifications_per_100k():
    detected = FlowRef("detection").sum() + FlowRef("acf_detection").sum()
    return 1e5 * detected / Reduce(sum_over=POP, where=PEOPLE)
```

`FlowRef` reads another flow's current value inside the vector field. Passing this to
`build_model(trigger_signal=...)` is the only change. The threshold is now 200 notifications
per 100,000 per year, which in the baseline is reached when prevalence is about 335.

The figure compares the two triggers. Notification-triggered ACF fires about a year earlier,
because its threshold corresponds to a lower prevalence. The notifications panel shows a
feedback that prevalence triggers do not have: **ACF itself raises notifications**. The first
sliver of programme activity adds ACF-detected cases, which pushes the signal over the
threshold slightly sooner, and the jump after triggering is large. That does no harm with a
latched trigger. For a *reactive* policy on notifications, though, the intervention would be
inflating its own trigger signal, which is worth knowing before designing one.

In [ ]:
by_notifications = tbm.build_model(tbm.notifications_per_100k()).compile()
notif = tbm.indicators(tbm.run(by_notifications, tbm.scenario_params("acf_triggered", threshold=200.0)))
by_prev = tbm.indicators(tbm.run(cm, tbm.scenario_params("acf_triggered")))

rows = []
for label, frame in {"baseline": baseline, "ACF triggered by prevalence ≥ 350": by_prev, "ACF triggered by notifications ≥ 200": notif}.items():
    for indicator in ("prevalence", "notifications", "programme_active"):
        rows.append(frame.loc[2020:2045, [indicator]].rename(columns={indicator: "value"}).assign(scenario=label, indicator=indicator).reset_index())
both = pd.concat(rows)
fig = both.plot(x="year", y="value", color="scenario", facet_row="indicator", height=750, title="Prevalence-triggered vs notification-triggered ACF")
fig.update_yaxes(matches=None)
fig

## Where to go from here

The trigger is three ordinary summer4 pieces: a `Reduce` or `FlowRef` signal, a `defer`-wrapped
switch, and a few bookkeeping compartments. Extensions follow the same pattern:

* **Stratified triggers.** With age or district strata, `Reduce(sum_over=district, ...)` gives
  prevalence per district, and a programme side model stratified by district can trigger each
  district separately.
* **Surveillance delay.** Put a chain of compartments between `scaling_up` and `active`, or
  between `waiting` and `scaling_up`, for an Erlang-distributed delay.
* **Calibration.** Everything above runs under `jit`, `vmap` and `grad`, so a triggered
  scenario can be run over posterior draws with summer4's calibration tools like any other
  scenario.